## Preamble
Run this cell on every new Spark notebook on Colab.

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
!java -version
!pip install pyspark

update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
openjdk version "1.8.0_275"
OpenJDK Runtime Environment (build 1.8.0_275-8u275-b01-0ubuntu1~18.04-b01)
OpenJDK 64-Bit Server VM (build 25.275-b01, mixed mode)
     |████████████████████████████████| 204.2MB 76kB/s 
     |████████████████████████████████| 204kB 47.8MB/s 
  Created wheel for pyspark: filename=pyspark-3.0.1-py2.py3-none-any.whl size=204612243 sha256=2f4944c28a760d6653df3e7986c8cd0b8a1a0b7fdbc116d749a7fcebc9b2986e
  Stored in directory: /root/.cache/pip/wheels/5e/bd/07/031766ca628adec8435bb40f0bd83bb676ce65ff4007f8e73f
Successfully built pyspark


# SPARK: Word Count

In [3]:
from pyspark.sql import SparkSession


In [4]:
spark = SparkSession.builder.master('local').appName('wordcount').getOrCreate()

In [5]:
sc = spark.sparkContext

In [6]:
!ls sample_data

anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


It's a good exercise to picture in mind what the RDD looks like at each and every step of the computation. To check the correctness you can run `rdd.take(2)` for get the first 2 elements. **BEWARE** this would be an `action` so in case of complex DAG you'll force the framework to materialize intermediate results (which can be costly).

In [7]:
lines = sc.textFile('sample_data/README.md')     # ["line1", "line2", ...... , "lineN"]  distributed collection of lines 

In [8]:
words = lines.flatMap(lambda x: x.split(' '))    # ["word1", "word2", ..., "wordN",  ]

This is how the `flatMap` method works. 

If we did `lines.map(lambda x: x.split(' '))` we would have the following:

$[[word_{1-1}, word_{1-2}, ... , word_{1-M}], [word_{2-1}, word_{2-2}, ... , word_{2-N}], ....]$ 

Where $word_{i-j}$ represents the $j^{th}$ word on the $i^{th}$ line.

Instead we *flattened* the "list of lists" to have the result.

In [9]:
word_count = words.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)

The RDD `word_count` is the result of two transformations. That is, two intermediate RDD are being created.

`first = words.map(lambda x: (x, 1)`

and:

`second = first.reduceByKey(lambda x, y: x + y)`

### First

`words` RDD is in the form `[w1, w2, ..., wN]`

So, `first` is in the form `[(w1, 1), (w2, 1), ..., (wN, 1)]`.


### Second

The `reduceByKey` transformation operates on key-value pairs.
   * key/values distributed collections
   * act on each key in parallel
   * regroup data across the network

So, what we have as an intermediate result would be something like (for example):

`intermediate = [(w1, [1,1,1]), (w2, [1]), ...]`

Thus, `second` is in the form `[(w1, 3), (w2, 1), ..., (wN, 4)]`


In [10]:
word_count.saveAsTextFile('outputFile')    # this is the ACTION. All the DAG is evaluated when you fire this cell. Before nothing happens.

In [11]:
!ls

outputFile  sample_data


In [12]:
!ls outputFile

part-00000  _SUCCESS


In [13]:
!cat outputFile/part-00000

('This', 1)
('directory', 1)
('includes', 1)
('a', 3)
('few', 1)
('sample', 2)
('datasets', 1)
('to', 1)
('get', 1)
('you', 1)
('started.', 1)
('', 51)
('*', 3)
('`california_housing_data*.csv`', 1)
('is', 4)
('California', 1)
('housing', 1)
('data', 1)
('from', 1)
('the', 3)
('1990', 1)
('US', 1)
('Census;', 1)
('more', 1)
('information', 1)
('available', 1)
('at:', 2)
('https://developers.google.com/machine-learning/crash-course/california-housing-data-description', 1)
('`mnist_*.csv`', 1)
('small', 1)
('of', 2)
('[MNIST', 1)
('database](https://en.wikipedia.org/wiki/MNIST_database),', 1)
('which', 1)
('described', 2)
('http://yann.lecun.com/exdb/mnist/', 1)
('`anscombe.json`', 1)
('contains', 1)
('copy', 2)
("[Anscombe's", 1)
('quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet);', 1)
('it', 1)
('was', 2)
('originally', 1)
('in', 2)
('Anscombe,', 1)
('F.', 1)
('J.', 1)
('(1973).', 1)
("'Graphs", 1)
('Statistical', 1)
("Analysis'.", 1)
('American', 1)
('Statistician.', 1)
('

In [14]:
word_count.take(3)

[('This', 1), ('directory', 1), ('includes', 1)]